# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/talhahmad-webdev/Machine-Learning-Engineering-internship--FlyrankAI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [76]:
!git clone https://github.com/talhahmad-webdev/Machine-Learning-Engineering-internship--FlyrankAI.git

fatal: destination path 'Machine-Learning-Engineering-internship--FlyrankAI' already exists and is not an empty directory.


In [77]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/Machine-Learning-Engineering-internship--FlyrankAI/data/raw/content_refresh_anonymized.csv"
)

print(df.shape)
df.columns.tolist()

(30000, 44)


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

## My Rule

A page should be refreshed if it has not been updated for a long time, has meaningful search demand, and is already receiving visibility in search results but still has room for improvement.

This rule prioritizes content that is more likely to benefit from a content refresh while avoiding pages with little search opportunity.

In [78]:
# Signal 1: Days Since Last Update

staleness_table = (
    df.assign(
        staleness_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[0, 90, 180, 365, df["days_since_last_update"].max()],
            labels=["0-90", "91-180", "181-365", "365+"],
            include_lowest=True
        )
    )
    .groupby("staleness_bucket")
    .agg(
        n=("content_id", "count"),
        avg_search_volume=("search_volume", "mean"),
        avg_ctr=("ctr", "mean")
    )
)

staleness_table

/tmp/ipykernel_1397/1925377514.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("staleness_bucket")


,n,avg_search_volume,avg_ctr
staleness_bucket,,,
0-90,20655,164.337421,0.604856
91-180,9171,148.585790,0.238367
181-365,169,40.375000,3.210828
365+,5,0.000000,20.000000


### Signal Check 1 – Days Since Last Update

**Verdict:** MIXED

**Reason:**
The relationship between staleness and search opportunity is not consistent. Very old content has too few examples (only 5 pages in the oldest bucket), making the averages unreliable. This suggests that days since last update alone is not a strong signal for prioritizing content refresh.

In [79]:
# Signal 2: Search Volume

volume_table = (
    df.assign(
        volume_bucket=pd.qcut(
            df["search_volume"],
            q=4,
            duplicates="drop"
        )
    )
    .groupby("volume_bucket")
    .agg(
        n=("content_id", "count"),
        avg_ctr=("ctr", "mean"),
        avg_position=("avg_position", "mean")
    )
)

volume_table

/tmp/ipykernel_1397/2947330656.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("volume_bucket")


,n,avg_ctr,avg_position
volume_bucket,,,
"(-0.001, 10.0]",18392,0.376270,16.387174
"(10.0, 20.0]",2290,0.252389,17.016114
"(20.0, 74000.0]",6850,0.195364,19.167358


### Signal Check 2 – Search Volume

**Reason**

Search volume represents the potential impact of refreshing a page. Pages with higher search demand can provide greater business value if improved.

**Verdict:** CONFIRMED

**Why**

The bucket analysis shows that higher search-volume pages generally have lower average CTR, suggesting there is more room for optimization despite strong search demand. This makes search volume a useful signal for prioritizing refresh opportunities.

### Reason Codes

- **high_demand_low_ctr**: The page has high search demand but a low CTR, making it a strong refresh candidate.

- **monitor**: The page does not currently meet the refresh criteria and should be monitored.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [80]:
import os

output_dir = "/content/Machine-Learning-Engineering-internship--FlyrankAI/work/outputs"
os.makedirs(output_dir, exist_ok=True)

In [81]:
# -----------------------------
# Baseline Score
# -----------------------------

df["baseline_score"] = 0

# Search demand
df["baseline_score"] += (df["search_volume"] > 10).astype(int)

# Page receives visibility
df["baseline_score"] += (df["impressions_90d"] > 0).astype(int)

# Already ranking
df["baseline_score"] += (
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20)
).astype(int)

# CTR below median = improvement opportunity
ctr_threshold = df["ctr"].median()

df["baseline_score"] += (
    df["ctr"] < ctr_threshold
).astype(int)

df["baseline_score"].value_counts().sort_index()

,count
baseline_score,
1,2356
2,13433
3,11860
4,2351


In [82]:
df["reason_code"] = np.where(
    df["baseline_score"] >= 3,
    "high_demand_low_ctr",
    "monitor"
)

In [83]:
df["action"] = np.where(
    df["baseline_score"] >= 3,
    "Refresh Content",
    "Monitor"
)

In [84]:
ranked_queue = (
    df.sort_values(
        by=[
            "baseline_score",
            "search_volume",
            "impressions_90d"
        ],
        ascending=[False, False, False]
    )
)

In [85]:
ranked_queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].to_csv(
    f"{output_dir}/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv written successfully.")

baseline_action_score.csv written successfully.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [86]:
top10 = ranked_queue.head(10)

top10[
    [
        "content_id",
        "search_volume",
        "impressions_90d",
        "avg_position",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

,content_id,search_volume,impressions_90d,avg_position,ctr,baseline_score,reason_code,action
5287,content_8ca50876b0df,40500.0,8699,18.3,0.03,4,high_demand_low_ctr,Refresh Content
2553,content_eb1510f4b5f1,33100.0,12275,14.7,0.00,4,high_demand_low_ctr,Refresh Content
16585,content_19bdaa296a9b,33100.0,359,13.5,0.00,4,high_demand_low_ctr,Refresh Content
9217,content_12e48d4b449d,27100.0,10,17.0,0.00,4,high_demand_low_ctr,Refresh Content
10466,content_71f8734aebe2,27100.0,1,15.0,0.00,4,high_demand_low_ctr,Refresh Content
18336,content_82f8b592944a,22200.0,6188,16.1,0.00,4,high_demand_low_ctr,Refresh Content
17859,content_5c5fab9d41e7,22200.0,1794,5.4,0.06,4,high_demand_low_ctr,Refresh Content
3752,content_e7eb94e121b9,22200.0,650,15.6,0.00,4,high_demand_low_ctr,Refresh Content
15616,content_201a4a56f4d6,22200.0,2,1.0,0.00,4,high_demand_low_ctr,Refresh Content
22507,content_aa6fbeaf8434,18100.0,2644,17.0,0.00,4,high_demand_low_ctr,Refresh Content


## Top-10 Review

### 1. content_8ca50876b0df
- **Action:** Refresh Content
- **Why it's there:** Very high search volume (40,500), strong impressions (8,699), low CTR (0.03), and received the highest baseline score.
- **What would make it wrong:** If the low CTR is caused by search intent mismatch instead of outdated content.

### 2. content_eb1510f4b5f1
- **Action:** Refresh Content
- **Why it's there:** High search volume (33,100), very high impressions (12,275), zero CTR, and baseline score of 4.
- **What would make it wrong:** If tracking issues or missing click data caused the zero CTR.

### 3. content_19bdaa296a9b
- **Action:** Refresh Content
- **Why it's there:** High search demand (33,100), low CTR, and baseline score of 4.
- **What would make it wrong:** The page has relatively low impressions (359), so demand may not translate into real visibility.

### 4. content_12e48d4b449d
- **Action:** Refresh Content
- **Why it's there:** High search volume (27,100), low CTR, and maximum baseline score.
- **What would make it wrong:** Only 10 impressions were recorded, which may not be enough evidence.

### 5. content_71f8734aebe2
- **Action:** Refresh Content
- **Why it's there:** High search demand (27,100) with zero CTR.
- **What would make it wrong:** Only one impression exists, making the recommendation less reliable.

### 6. content_82f8b592944a
- **Action:** Refresh Content
- **Why it's there:** High search volume (22,200), strong impressions (6,188), zero CTR, and score of 4.
- **What would make it wrong:** If users are intentionally not clicking because the search result already answers their question.

### 7. content_5c5fab9d41e7
- **Action:** Refresh Content
- **Why it's there:** High search volume (22,200), good visibility (1,794 impressions), low CTR (0.06), and maximum score.
- **What would make it wrong:** A good ranking (5.4) may indicate the content itself is not the primary issue.

### 8. content_e7eb94e121b9
- **Action:** Refresh Content
- **Why it's there:** High search volume (22,200), measurable impressions (650), zero CTR, and high baseline score.
- **What would make it wrong:** The CTR issue could be caused by an unattractive title or meta description rather than outdated content.

### 9. content_201a4a56f4d6
- **Action:** Refresh Content
- **Why it's there:** High search demand and maximum baseline score.
- **What would make it wrong:** Only two impressions were recorded, so there is not enough evidence for a confident recommendation.

### 10. content_aa6fbeaf8434
- **Action:** Refresh Content
- **Why it's there:** High search volume (18,100), good impressions (2,644), zero CTR, and baseline score of 4.
- **What would make it wrong:** Low CTR may be due to factors outside the page content, such as SERP competition.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

The baseline rule successfully identifies pages with high search demand and low CTR, but there are some weak recommendations.

1. Some pages have extremely low impressions (for example, 1–10 impressions) but still receive the maximum baseline score because of their high search volume.

2. The rule does not distinguish whether a low CTR is caused by outdated content or by other factors such as poor titles, meta descriptions, or strong SERP competition.

3. Some pages already have good average positions, so refreshing the content may not be the most effective action.

Overall, this baseline is simple and transparent, but it may produce false positives. A machine learning model in Week 5 should improve the ranking by learning more complex patterns from multiple signals.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.